<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 7 &nbsp;&middot;&nbsp; Large Language Models</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>What LLMs are, how they understand language, and how to use them through APIs to build real business applications &mdash; without training a single model yourself.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| 7.1 What Are LLMs? | From word2vec to Transformers · LLMs vs SLMs · Why this changed everything |
| 7.2 Tokenization and Embeddings | How text becomes numbers · BPE · Hugging Face tokenizers · Semantic similarity |
| 7.3 Using LLM APIs | First API call · Prompt engineering · Structured outputs |
| 7.4 Building a Chat App | Conversation history · System prompts · Gradio interface |
| 7.5 Ethical Considerations | Hallucination · Bias · Privacy · Cost |
| 7.6 Capstone: Business Q&A Bot | RAG concept · Document chunking · Full Q&A pipeline |

> **No local training in this chapter.** You will direct powerful pre-trained models through APIs.
> A free API key from OpenAI, Google Gemini, or Anthropic Claude is all you need.
> No GPU required.

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 7 — Setup
# Installs everything this chapter needs.
# Expected time: 2–3 minutes on first run.
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet transformers gradio openai google-generativeai anthropic

import os
import json
import textwrap
from transformers import AutoTokenizer

print("Setup complete. Ready to begin Chapter 7.")

---
## Where We Are Coming From

In Chapters 1 through 6, you built neural networks from scratch.

- In **Chapter 3**, you trained a regression network and saved it as a `.pth` file.
- In **Chapter 4**, you trained a classifier to predict customer churn.
- In **Chapter 5**, you trained a CNN to detect defects in product images.
- In **Chapter 6**, you trained an LSTM to forecast CO₂ levels weeks into the future.

Every time, the process was the same: collect data, design an architecture, run a training loop, save the weights.

**Chapter 7 is different.**

The models you will use here — Large Language Models — were trained by teams of hundreds of engineers on hundreds of billions of words, using thousands of GPUs, over months. You are not going to train them. You are going to *direct* them.

Think of the shift like this:

> **Chapters 1–6:** You baked the cake from scratch — flour, sugar, eggs, oven, timing.
>
> **Chapter 7:** A professional bakery already made the cake. Your job is to tell them exactly how you want it sliced, decorated, and served.

This is not a step down in skill. Knowing how to give clear, precise instructions to a powerful model — and knowing how to build applications around it — is one of the most valuable skills in business analytics today.

---
# 7.1 What Are Large Language Models?

## Starting simple: what does a language model do?

A **language model** is a system that has learned to predict what word comes next in a sentence.

That sounds almost too simple. But if you predict the next word well enough, across enough text, something remarkable happens: the model develops a deep working understanding of grammar, facts, logic, tone, and meaning — all without ever being told what any of those things are.

**Example:**

> "The capital of France is ___"

A language model that has seen enough text will fill in "Paris" with very high confidence. It has never been told the definition of "capital city." It learned the relationship from patterns in billions of sentences.

> "The meeting was cancelled because the CEO was ___ with the results."

Here the model might predict "unhappy", "dissatisfied", or "furious" — all contextually appropriate. It has learned how professional language works.

---

## A brief history — in plain language

```
Word2Vec (2013)
     |
     |  Words as vectors. "King" - "Man" + "Woman" ~ "Queen"
     |  Each word had ONE fixed representation.
     v
RNNs and LSTMs  (Chapter 6)
     |
     |  Text processed word-by-word with a hidden state.
     |  Good — but struggled with long documents.
     v
Transformers (2017 — "Attention Is All You Need")
     |
     |  Every word attends to every other word simultaneously.
     |  No recurrence — fully parallel. Scales beautifully.
     v
GPT, BERT, T5 (2018–2020)
     |
     |  Pre-trained on massive corpora.
     |  Fine-tuned on specific tasks.
     v
GPT-4 / Gemini / Claude / Llama (2020–present)
     |
     |  Hundreds of billions of parameters.
     |  Instruction-following, reasoning, coding,
     |  summarisation, translation — from one model.
     v
You are here.
```

---

## What makes an LLM "large"?

Three things working together:

| Ingredient | What it means | Example scale |
|-----------|---------------|---------------|
| **Parameters** | The learned numbers inside the model | GPT-4: estimated ~1 trillion |
| **Training data** | Text the model learned from | Books, websites, code, scientific papers |
| **Compute** | The GPU-hours needed to train it | Millions of dollars worth |

You do not need to understand attention mechanics to use an LLM effectively. But it is useful to know that the model has, in effect, read more text than any human ever could — and compressed what it learned into its weights.

---

## LLMs vs. SLMs — which one do you need?

| | **LLM** (Large Language Model) | **SLM** (Small Language Model) |
|-|-------------------------------|-------------------------------|
| **Examples** | GPT-4o, Gemini 1.5 Pro, Claude 3.5 | GPT-4o-mini, Gemini Flash, Phi-3 |
| **Strengths** | Complex reasoning, nuanced writing, coding | Speed, cost, privacy, on-device use |
| **Best for** | Multi-step analysis, creative tasks | Classification, extraction, simple Q&A |
| **Cost** | Higher per 1,000 tokens | Much lower |

**Rule of thumb:** Start with a smaller, cheaper model. If the quality is good enough, you are done. Only move to a larger model when the task genuinely requires it.

---
### 💡 Illustration: Why Attention Changed Everything

Imagine you are reading the sentence:

> *"The bank by the river was steep, so the fisherman tied his boat to the **bank**."*

The second "bank" means riverbank, not financial institution. How do you know?
Because you attended to "river", "fisherman", and "boat" earlier in the sentence.

This is exactly what a Transformer's **attention mechanism** does — for every word,
it learns to look at every other word and decide which ones are most relevant.

```
Sentence: "The bank by the river was steep"

When processing "bank", the model attends to:

    "The"    --> low attention  (0.02)
    "bank"   --> self           (0.15)
    "by"     --> low            (0.03)
    "the"    --> low            (0.02)
    "river"  --> HIGH attention (0.61)  <-- this is the key signal
    "was"    --> low            (0.05)
    "steep"  --> medium         (0.12)
```

By attending strongly to "river", the model learns that this "bank" is a geographical
feature, not a financial one.

An LSTM (Chapter 6) had to carry this context forward one step at a time.
A Transformer reads the whole sentence at once and figures out which parts matter.
That is why Transformers work better on long documents.

### 📝 Exercise 7.1 — LLMs in Your Organisation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.1
#
# Think about the organisation or industry you work in (or want to work in).
# Identify THREE tasks where an LLM could save 10+ hours per week.
#
# For each task, write:
#   Task       : what the person currently does manually
#   LLM action : what the LLM would do instead
#   Time saved : rough estimate per week
#   Risk       : one thing that could go wrong
#
# Example:
#   Task       : A lawyer reads 50 contracts per week looking for unusual clauses
#   LLM action : LLM flags unusual clauses with a brief explanation
#   Time saved : ~15 hours/week
#   Risk       : LLM misses a subtle clause; lawyer still needs to review
#
# Task 1:
#   Task       : ?
#   LLM action : ?
#   Time saved : ?
#   Risk       : ?
#
# Task 2:
#   Task       : ?
#   LLM action : ?
#   Time saved : ?
#   Risk       : ?
#
# Task 3:
#   Task       : ?
#   LLM action : ?
#   Time saved : ?
#   Risk       : ?
# ─────────────────────────────────────────────────────────────────────────────
print("Exercise 7.1 complete.")

---
# 7.2 Tokenization and Embeddings

## How does text become numbers?

Neural networks work with numbers. Text is made of letters and words.
The conversion happens in two steps: **tokenization** then **embeddings**.

---

## Step 1: Tokenization — cutting text into pieces

A **token** is the basic unit that an LLM works with. Tokens are not always full words:

- A complete word: `"business"` → one token
- Part of a word: `"tokenization"` → might split into `["token", "ization"]`
- A punctuation mark: `"."` → one token

**Why split words?** A vocabulary of 50,000 tokens can cover almost any English text
by combining common sub-word pieces — including rare words, names, and technical terms.

---

### 💡 Illustration: Tokenization in action

```
Input: "The quarterly revenue declined unexpectedly."

Tokens:  ["The", "quarterly", "revenue", "declined", "unexpected", "##ly", "."]
IDs:     [  464,     18742,     7168,      3830,        8320,       ##ly,   13]

A less common word splits further:
  "unexpectedly" --> ["un", "expect", "ed", "ly"]   (4 tokens, not 1)
```

**Important for business users:** You are often charged per token.
Rough rule: 1 token ≈ 0.75 English words. 1,000 tokens ≈ one page of text.

---

## Step 2: Embeddings — turning tokens into meaning

After tokenization, each token gets converted to a **vector** — a list of numbers.

You saw this idea in Chapter 6 when we gave each of 52 weeks its own 8-number
vector using `nn.Embedding`. LLMs do the same for words, at much larger scale:

```
Chapter 6 (week embeddings):
   week_index --> 8-number vector    (52 possible weeks)

LLM (word embeddings):
   token_id   --> 768-number vector  (50,000 possible tokens)
               or 4096-number vector for larger models
```

The key property: **similar meanings end up close together in space.**

```
"revenue"  is close to  "income", "sales", "earnings"
"revenue"  is far from  "weather", "bicycle", "ocean"

"declined" is close to  "dropped", "fell", "decreased"
"declined" is far from  "improved", "grew", "increased"
```

This means the model understands that "sales fell" and "revenue declined"
are saying roughly the same thing — even though no word is shared.

### Tokenizing business text with Hugging Face

Let us see tokenization in action using a real tokenizer.
We use `bert-base-uncased` — a well-known model. Only the vocabulary file
is downloaded (~500 KB), not the full model.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tokenization: watching text become token IDs
# Only the vocabulary is downloaded -- not the full model.
# ─────────────────────────────────────────────────────────────────────────────

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sentences = [
    "The quarterly revenue declined unexpectedly.",
    "Our customer churn rate increased by 12 percent.",
    "The CEO announced a strategic restructuring plan.",
    "Tokenization splits uncommon words like synergistic carefully.",
]

print("=" * 60)
print("Tokenization Examples")
print("=" * 60)

for sentence in sentences:
    tokens    = tokenizer.tokenize(sentence)
    token_ids = tokenizer.encode(sentence, add_special_tokens=False)
    print(f"\nOriginal : {sentence}")
    print(f"Tokens   : {tokens}")
    print(f"IDs      : {token_ids}")
    print(f"Count    : {len(tokens)} tokens  |  Words: {len(sentence.split())}")
    print("-" * 60)

print("\nNotice:")
print("  - Common words are usually one token.")
print("  - Uncommon or long words often split into sub-word pieces.")
print("  - Punctuation becomes its own token.")
print("  - Token count > word count for rare/long words.")

### 📝 Exercise 7.2 — Explore Your Domain's Vocabulary

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.2 — Tokenize domain-specific business text
#
# Replace the sentences below with text from YOUR industry or field.
# Examples to try:
#   Legal   : "The indemnification clause supersedes the arbitration agreement."
#   Medical : "The patient exhibited tachycardia and diaphoresis post-operatively."
#   Finance : "The collateralised debt obligation was marked to market at par."
#   Tech    : "The microservices architecture enables horizontal scalability."
#
# After running, reflect:
#   1. Which words from your domain split into multiple tokens?
#   2. Why might this matter when estimating API costs?
#   3. Are there terms the tokenizer handles well vs. poorly?
# ─────────────────────────────────────────────────────────────────────────────

my_sentences = [
    "Replace this with a sentence from your industry.",
    "Add a second sentence with domain-specific terminology.",
    "Try a sentence with abbreviations like KPI, ROI, or EBITDA.",
]

print("=" * 60)
print("Your Domain Tokenization")
print("=" * 60)

for sentence in my_sentences:
    tokens = tokenizer.tokenize(sentence)
    print(f"\nSentence : {sentence}")
    print(f"Tokens   : {tokens}")
    print(f"Count    : {len(tokens)} tokens from {len(sentence.split())} words")
    print("-" * 60)

---
# 7.3 Using LLM APIs

## What is an API?

An **API** (Application Programming Interface) is a way for your code to talk to
another service over the internet.

When you make an LLM API call, you are saying:

> "Here is my text (the prompt). Please process it and send back the response."

You send text. You receive text back. No training, no GPU, no model files.

---

## The anatomy of an LLM API call

```
API CALL STRUCTURE
==================

1. MODEL       Which model to use
               "gpt-4o-mini", "gemini-1.5-flash", etc.

2. MESSAGES    The conversation so far
               role: "system"    --> instructions about how to behave
               role: "user"      --> what you are asking
               role: "assistant" --> what the model said previously

3. PARAMETERS
               temperature = 0   --> focused, deterministic
               temperature = 1   --> more creative and varied
               max_tokens        --> maximum length of the response
```

---

## Prompt engineering: the skill of giving good instructions

### Principle 1: Be specific

VAGUE:    "Summarise this email."

BETTER:   "Summarise this email in 3 bullet points. Each bullet should be
           one sentence. Focus on action items and deadlines."

### Principle 2: Give the model a role

NO ROLE:  "Is this contract clause problematic?"

BETTER:   "You are a contracts analyst at a mid-size technology company.
           Review the following clause and flag any unusual terms that might
           increase our liability. Be concise."

### Principle 3: Show the format you want

NO FORMAT: "Classify this customer complaint."

BETTER:    "Classify the complaint. Respond only with a JSON object in this
            format: {category, urgency, summary}"

### Principle 4: Give examples when the task is ambiguous

"Here are two examples of how I want you to classify emails:
 Input: 'When will my order arrive?' --> Output: 'order_status'
 Input: 'I want a refund'            --> Output: 'refund_request'
 Now classify: 'My package was damaged.'"

---

## Setting up your API key

Choose any one of the following — all have free tiers sufficient for this chapter:

| Provider | Get a free key at | Model to use |
|----------|------------------|--------------|
| OpenAI | platform.openai.com | `gpt-4o-mini` |
| Google Gemini | aistudio.google.com | `gemini-1.5-flash` |
| Anthropic Claude | console.anthropic.com | `claude-haiku-4-5` |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# API Key Setup
#
# Paste your API key below.
# On Colab: you can also use the Secrets panel (key icon in the left sidebar):
#   Add a secret named OPENAI_API_KEY and enable notebook access.
# ─────────────────────────────────────────────────────────────────────────────

import os

# Option A: paste directly (simpler for learning)
API_KEY    = "YOUR_API_KEY_HERE"   # <-- replace with your key
MODEL_NAME = "gpt-4o-mini"        # or "gemini-1.5-flash" or "claude-haiku-4-5"

# Option B: load from Colab Secrets (more secure)
# from google.colab import userdata
# API_KEY = userdata.get("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = API_KEY
print("API key set. Ready to make your first API call.")

### Your first API call

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# First API call -- a simple business question
# Read through every comment: the pattern is explained line by line.
# ─────────────────────────────────────────────────────────────────────────────

from openai import OpenAI

client = OpenAI(api_key=API_KEY)

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        # System message: tells the model HOW to behave
        {
            "role": "system",
            "content": (
                "You are a helpful business analyst. "
                "Give concise, practical answers. "
                "Use bullet points when listing multiple items."
            )
        },
        # User message: the actual question
        {
            "role": "user",
            "content": "What are three early warning signs that a company's customer churn rate is about to increase?"
        }
    ],
    temperature=0.3,   # low = more focused and consistent
    max_tokens=300,
)

answer = response.choices[0].message.content

print("=" * 60)
print("Model response:")
print("=" * 60)
print(answer)
print()
print(f"Tokens -- Prompt: {response.usage.prompt_tokens} | "
      f"Response: {response.usage.completion_tokens} | "
      f"Total: {response.usage.total_tokens}")

### Structured outputs — getting JSON back

In business applications, you often want the model to return structured data
(a category, a score, a list of fields) rather than free-form prose.
You can do this by being explicit in your prompt.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Structured output: classifying customer complaints as JSON
# temperature=0.0 for maximum consistency in classification tasks
# ─────────────────────────────────────────────────────────────────────────────

import json

customer_complaints = [
    "My order arrived three weeks late and the packaging was destroyed.",
    "I have been charged twice for the same subscription this month.",
    "Your app keeps crashing every time I try to view my account balance.",
    "The product works fine but I could not find the return policy anywhere.",
]

CLASSIFICATION_SYSTEM_PROMPT = (
    "You are a customer service analyst. "
    "Classify each complaint and respond ONLY with a JSON object. "
    "No explanation, no extra text -- only the JSON. "
    "Format: "
    '{"category": "delivery|billing|technical|information", '
    '"urgency": "low|medium|high", '
    '"sentiment": "neutral|frustrated|angry", '
    '"summary": "one sentence"}'
)

print("=" * 60)
print("Classifying customer complaints")
print("=" * 60)

for complaint in customer_complaints:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": CLASSIFICATION_SYSTEM_PROMPT},
            {"role": "user",   "content": complaint}
        ],
        temperature=0.0,
        max_tokens=150,
    )

    raw = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(raw)
        print(f"\nComplaint : {complaint[:65]}...")
        print(f"Category  : {parsed.get('category', 'n/a')}")
        print(f"Urgency   : {parsed.get('urgency', 'n/a')}")
        print(f"Sentiment : {parsed.get('sentiment', 'n/a')}")
        print(f"Summary   : {parsed.get('summary', 'n/a')}")
    except json.JSONDecodeError:
        print(f"\nComplaint : {complaint[:65]}")
        print(f"Raw output: {raw}")
        print("(Could not parse as JSON -- check prompt.)")

    print("-" * 60)

### 📝 Exercise 7.3 — Compare Two Temperature Settings

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.3 -- Compare temperature=0.0 vs temperature=0.9
#
# Use the same prompt twice with different temperature settings.
# After running, reflect:
#   1. Did both runs make the same key points?
#   2. Which felt more reliable for a business audience?
#   3. For a safety-critical task, which temperature would you choose?
# ─────────────────────────────────────────────────────────────────────────────

business_prompt = (
    "A mid-size retail company is considering replacing its human customer "
    "service team (20 agents) with an LLM-powered chatbot. "
    "What are the three most important risks they should evaluate? "
    "Respond in 3 numbered points, each 2-3 sentences."
)

print("Run A -- temperature 0.0 (focused):")
print("-" * 50)
r_a = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": business_prompt}],
    temperature=0.0, max_tokens=400,
)
print(r_a.choices[0].message.content)

print("\n\nRun B -- temperature 0.9 (more varied):")
print("-" * 50)
r_b = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": business_prompt}],
    temperature=0.9, max_tokens=400,
)
print(r_b.choices[0].message.content)

---
# 7.4 Building a Chat App

## Why conversation history matters

An LLM has **no memory between separate API calls**. Every call starts fresh.

You give it a conversation history by including all previous messages in the
`messages` list of each new call:

```
Turn 1 -- You send:
  messages = [
    {"role": "user", "content": "What is churn rate?"}
  ]

Turn 2 -- You send:
  messages = [
    {"role": "user",      "content": "What is churn rate?"},     <- previous
    {"role": "assistant", "content": "Churn rate is the..."},    <- previous
    {"role": "user",      "content": "How do I reduce it?"}      <- new
  ]
```

Each API call sends the full conversation. The model reads it all and
responds in context. This is how every chatbot is built.

---

## The system prompt: your model's personality and rules

The **system prompt** (`role: "system"`) appears at the start of every
conversation. It tells the model its job, tone, and limits.

Example system prompts:

```
# Financial report summariser:
"You are a financial analyst assistant. Summarise documents clearly.
Always include key numbers. Flag risks. Do not make recommendations."

# Customer service agent:
"You are a friendly representative for TechCorp. You can answer questions
about our products. If you do not know, say so and offer to escalate."

# Internal HR assistant:
"Answer questions about company policy based only on the provided documents.
If a question falls outside policy, say: I do not have policy on that topic."
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# A multi-turn chat loop
#
# We keep a running history list. Each new message is appended, then
# the full history is sent with every API call. This is the production pattern.
# ─────────────────────────────────────────────────────────────────────────────

def chat(system_prompt, user_questions):
    history = [{"role": "system", "content": system_prompt}]

    print("=" * 60)
    print("Chat Session")
    print("=" * 60)

    for question in user_questions:
        history.append({"role": "user", "content": question})

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=history,
            temperature=0.4,
            max_tokens=250,
        )

        reply = response.choices[0].message.content
        history.append({"role": "assistant", "content": reply})

        print(f"\nYou       : {question}")
        print(f"Assistant : {reply}")
        print("-" * 60)

    print(f"\nTotal turns: {len(user_questions)}")
    print(f"Messages in final history: {len(history)}")


BUSINESS_PROMPT = (
    "You are a concise business analytics assistant. "
    "You help analysts interpret data and metrics. "
    "Keep answers to 3 sentences or fewer unless more detail is requested."
)

conversation = [
    "Our monthly active users dropped 15% last quarter. What could be causing this?",
    "Which of those causes is most likely to respond to a marketing campaign?",
    "What metric should I track to know if the campaign worked?",
]

chat(BUSINESS_PROMPT, conversation)

### Adding a Gradio interface

To share your chatbot as a real web app, wrap it in **Gradio** — a Python
library that turns a function into an interactive web page in one step.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Gradio chat interface
#
# Gradio generates a public URL anyone can open in a browser.
# The URL is temporary (72 hours) and free.
#
# How it works:
#   - Gradio calls respond() on each new message
#   - We maintain history in Gradio's state
#   - Gradio handles all the UI; we write only the logic
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr

SYSTEM_PROMPT = (
    "You are a helpful business analytics assistant. "
    "Give concise, practical answers. "
    "When listing items, use bullet points. "
    "Always be professional and constructive."
)

def respond(user_message, chat_history):
    # Convert Gradio history format --> OpenAI messages format
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user_turn, assistant_turn in chat_history:
        messages.append({"role": "user",      "content": user_turn})
        messages.append({"role": "assistant", "content": assistant_turn})
    messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0.4,
        max_tokens=400,
    )
    reply = response.choices[0].message.content
    chat_history.append((user_message, reply))
    return "", chat_history

with gr.Blocks(title="Business Analytics Assistant") as demo:
    gr.Markdown("## Business Analytics Assistant")
    gr.Markdown("Ask anything about data, metrics, or business strategy.")

    chatbot  = gr.Chatbot(height=400)
    msg_box  = gr.Textbox(placeholder="Type your question here...", label="Your message")
    send_btn = gr.Button("Send", variant="primary")
    clear    = gr.Button("Clear conversation")

    send_btn.click(respond, [msg_box, chatbot], [msg_box, chatbot])
    msg_box.submit(respond, [msg_box, chatbot], [msg_box, chatbot])
    clear.click(lambda: ([], ""), None, [chatbot, msg_box])

print("Starting Gradio app...")
print("A public share URL will appear below.")
demo.launch(share=True)

---
# 7.5 Ethical Considerations

## You now have a powerful tool. Here is what can go wrong.

---

## Risk 1: Hallucination

An LLM will sometimes **confidently state something that is completely false**.

The model does not "know" things the way a database does. It generates the most
statistically plausible next token — and sometimes the most plausible thing is wrong.

```
EXAMPLE:

Prompt:  "List five studies showing that Product X reduces churn."
Model:   "1. Johnson & Williams (2021) found that Product X reduced
              churn by 23% in SaaS companies..."

These studies do not exist. The model invented them, confidently.
```

**The rule:** Never trust an LLM for facts without verification.
Use LLMs for *generating*, *structuring*, and *reasoning* — not as a source of truth.

---

## Risk 2: Bias

LLMs learned from human-written text. Human text reflects human biases.
The model will reflect those biases in its outputs.

This matters when screening job applications, scoring customer sentiment by
demographic, or making recommendations about people.

**The rule:** Test prompts across diverse examples. Check for systematic
differences across gender, ethnicity, age, and other attributes.

---

## Risk 3: Data Privacy

When you send text to an API, it leaves your organisation and travels to
someone else's servers.

**Never send:** customer names or PII, internal financial data, personal health
information, passwords, or API keys.

**Do instead:** Anonymise before sending. Check whether your provider uses
your data to train future models (most offer an opt-out).

---

## Risk 4: Cost Awareness

API costs are measured in **tokens**. A quick cost estimate:

```
Task: Summarise 1,000 customer support tickets per day
  Average ticket:  200 words ~ 270 input tokens
  Average summary:  50 words ~  70 output tokens

At $0.15 / 1M input tokens + $0.60 / 1M output tokens:
  Input:  1,000 x 270 x $0.15 / 1,000,000 = $0.04/day
  Output: 1,000 x  70 x $0.60 / 1,000,000 = $0.04/day
  Total: ~$0.08/day = ~$2.40/month (very affordable)
```

Always prototype with the cheapest model.
Scale up only when quality genuinely requires it.

### 📝 Exercise 7.5 — Deliberately Prompt a Hallucination

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.5 -- Observe hallucination in action
#
# You need to SEE the failure mode, not just read about it.
# The prompts below ask for things that are likely to trigger hallucination.
#
# After running, reflect:
#   1. Did the model admit uncertainty, or did it sound confident?
#   2. If you did not already know the answer, would you have believed it?
#   3. What kind of verification would you add before using this output?
# ─────────────────────────────────────────────────────────────────────────────

hallucination_prompts = [
    (
        "Citation trap",
        "Cite three peer-reviewed studies from 2022-2023 that specifically examined "
        "the effect of 4-day work weeks on customer satisfaction scores in retail banking."
    ),
    (
        "Specific statistics trap",
        "What was the exact customer churn rate for mid-size SaaS companies "
        "in Southeast Asia in Q3 2023?"
    ),
    (
        "Fictional regulation trap",
        "Summarise the key requirements of the EU Digital Analytics Compliance "
        "Directive 2023/847 for companies processing consumer behavioural data."
    ),
]

print("=" * 60)
print("Hallucination Exercise")
print("=" * 60)

for label, prompt in hallucination_prompts:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300,
    )
    reply = response.choices[0].message.content
    print(f"\n[{label}]")
    print(f"Prompt : {prompt[:90]}...")
    print(f"\nModel response:")
    print(reply)
    print("-" * 60)

print("\nRemember: confident tone does not mean correct answer.")
print("Always verify facts from LLM outputs against authoritative sources.")

---
# 7.6 Capstone: Building a Business Q&A Bot

## The problem with general LLMs in business

A general LLM does not know *your* documents.

If you ask: "What did our CEO say about expansion plans in the last earnings call?"
The model will either make something up, or say it has no access to that information.
Neither is useful.

The solution is **RAG — Retrieval-Augmented Generation**.

---

## What is RAG?

RAG is a two-step approach:

```
STEP 1: RETRIEVE
  Your documents            User asks:
  (earnings calls,          "What did the CEO say about growth?"
   policy docs,
   FAQs, reports)  -------> Search for the most relevant
                            passages in your documents
          |
          v
STEP 2: GENERATE
  Take the retrieved passages + the user's question
  and send them together to the LLM:

  "Using only the text below, answer the question.
   Text: [retrieved passages]
   Question: What did the CEO say about growth?"

  The model answers based on YOUR documents -- not its training.
```

This solves the hallucination problem for factual questions about your documents:
the model is grounded in text you provided, and you can verify every answer.

---

## Building the Q&A bot

The pipeline:

```
Documents --> Split into chunks --> Search chunks --> Answer with context
```

For this capstone we use keyword search instead of vector embeddings.
The logical flow is identical to a production RAG system.
Switching to semantic embeddings later is a straightforward upgrade.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1 -- Load the document corpus
#
# Fictional earnings call excerpts.
# In your real project, replace with:
#   - PDF annual reports (use pdfplumber to extract text)
#   - Policy documents as plain text files
#   - Any text you have permission to use
# ─────────────────────────────────────────────────────────────────────────────

CORPUS = [
    {
        "source": "Q3 2024 Earnings Call -- CEO Opening",
        "text": (
            "Our Q3 revenue came in at $248 million, representing 18% year-over-year growth. "
            "We are particularly encouraged by performance in our enterprise segment, which "
            "grew 31% and now represents 44% of total revenue. Customer churn in the SMB "
            "segment remains an area of focus -- we saw churn tick up to 3.2% this quarter "
            "from 2.8% last quarter. We attribute this primarily to macroeconomic pressure "
            "on smaller businesses and are implementing targeted retention programmes."
        )
    },
    {
        "source": "Q3 2024 Earnings Call -- CFO Remarks",
        "text": (
            "Gross margin for the quarter was 71.4%, up from 69.8% in Q3 last year, driven "
            "by infrastructure efficiency gains and a higher mix of enterprise contracts. "
            "Operating expenses increased 12% year-over-year, primarily in sales and marketing "
            "as we invest in our new regional expansion in Southeast Asia. Free cash flow was "
            "$42 million, our strongest quarter to date. We are raising full-year guidance "
            "to $960-970 million in revenue."
        )
    },
    {
        "source": "Q3 2024 Earnings Call -- Analyst Q&A",
        "text": (
            "Analyst: Can you comment on the competitive landscape in the APAC market? "
            "CEO: We have seen the new entrant's launch and take all competition seriously. "
            "Our enterprise relationships and 8-year track record of data security compliance "
            "give us significant advantages. We have not seen meaningful customer losses to "
            "this competitor as of this quarter. "
            "Analyst: What is your outlook for churn in Q4? "
            "CFO: We expect SMB churn to stabilise at around 3.0-3.2% as macro conditions "
            "improve. Our new customer success programme should show measurable impact by Q1 2025."
        )
    },
    {
        "source": "Q2 2024 Earnings Call -- CEO Opening",
        "text": (
            "Q2 revenue was $220 million, up 21% year-over-year. This was our first quarter "
            "exceeding $200 million, a significant milestone for the company. We added 1,200 "
            "net new enterprise customers, up from 890 in Q2 2023. Our new product line, "
            "launched in April, contributed $8 million in its first full quarter. Customer "
            "satisfaction scores reached an all-time high of 4.7 out of 5.0 in our annual survey."
        )
    },
    {
        "source": "2024 Annual Investor Presentation",
        "text": (
            "Our five-year strategic plan centres on three pillars: enterprise expansion, "
            "product platform consolidation, and geographic diversification. We target "
            "reaching $1.5 billion in annual recurring revenue by 2027, with 60% from "
            "enterprise accounts. Key risk factors include macroeconomic sensitivity of "
            "our SMB customer base, increasing competition, and the need to invest heavily "
            "in AI capabilities. We are committed to maintaining gross margins above 70%."
        )
    },
]

print(f"Corpus loaded: {len(CORPUS)} documents")
for i, doc in enumerate(CORPUS):
    print(f"  [{i}] {doc['source']} -- {len(doc['text'].split())} words")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 2 -- Chunk the documents
#
# Why chunk?
# LLMs have a context window -- a maximum amount of text they can process.
# Chunking lets us retrieve only the relevant parts, not the whole document.
#
# Strategy: ~150-word chunks with 20-word overlap at boundaries.
# ─────────────────────────────────────────────────────────────────────────────

def chunk_text(text, source, target_words=150, overlap_words=20):
    words  = text.split()
    chunks = []
    start  = 0
    chunk_id = 0

    while start < len(words):
        end = min(start + target_words, len(words))
        chunks.append({
            "source":   source,
            "chunk_id": chunk_id,
            "text":     " ".join(words[start:end])
        })
        if end == len(words):
            break
        start    += target_words - overlap_words
        chunk_id += 1

    return chunks

all_chunks = []
for doc in CORPUS:
    all_chunks.extend(chunk_text(doc["text"], doc["source"]))

print(f"Total chunks: {len(all_chunks)}")
for chunk in all_chunks[:3]:
    print(f"\n[{chunk['source']} | chunk {chunk['chunk_id']}]")
    print(f"  {chunk['text'][:100]}...")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 3 -- Retrieve relevant chunks for a question
#
# Production RAG uses vector embeddings and cosine similarity.
# This version uses keyword matching -- simpler, but the logical flow
# is identical. Adding embeddings is a straightforward upgrade later.
# ─────────────────────────────────────────────────────────────────────────────

def retrieve_chunks(question, chunks, top_k=3):
    stopwords = {
        "the", "a", "an", "is", "are", "was", "were", "what", "how", "why",
        "when", "did", "do", "does", "our", "in", "of", "to", "for", "and",
        "or", "it", "its", "about", "this", "that", "be", "at", "by"
    }
    question_words = set(question.lower().split()) - stopwords

    scored = []
    for chunk in chunks:
        chunk_words = set(chunk["text"].lower().split())
        overlap = len(question_words & chunk_words)
        scored.append((overlap, chunk))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [ch for score, ch in scored[:top_k] if score > 0]


test_questions = [
    "What was the churn rate this quarter?",
    "How did gross margin change year over year?",
    "What is the company's revenue target for 2027?",
]

print("Retrieval test:")
print("=" * 60)
for q in test_questions:
    retrieved = retrieve_chunks(q, all_chunks, top_k=2)
    print(f"\nQuestion: {q}")
    for r in retrieved:
        print(f"  [{r['source']}] {r['text'][:80]}...")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 4 -- The full Q&A pipeline: retrieve + answer
#
# Complete RAG loop:
#   1. User asks a question
#   2. Retrieve relevant chunks
#   3. Send chunks + question to the LLM
#   4. LLM answers based ONLY on the provided context
# ─────────────────────────────────────────────────────────────────────────────

def answer_question(question, chunks, client, model_name, top_k=3):
    relevant = retrieve_chunks(question, chunks, top_k=top_k)

    if not relevant:
        return {
            "answer":  "I could not find relevant information in the documents to answer this question.",
            "sources": [],
        }

    context_parts = []
    sources_used  = []
    for chunk in relevant:
        context_parts.append(f"[Source: {chunk['source']}]\n{chunk['text']}")
        if chunk["source"] not in sources_used:
            sources_used.append(chunk["source"])

    context = "\n\n".join(context_parts)

    system_prompt = (
        "You are a financial analyst assistant. "
        "Answer the user's question using ONLY the context provided below. "
        "If the context does not contain enough information, say so clearly -- "
        "do not guess or add external information. "
        "Always mention which document your answer comes from."
    )

    user_prompt = f"Context:\n{context}\n\nQuestion: {question}"

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=400,
    )

    return {
        "answer":  response.choices[0].message.content,
        "sources": sources_used,
    }


questions = [
    "What was the SMB churn rate in Q3 2024 and what caused it?",
    "What is the company's gross margin and how has it changed?",
    "What are the three pillars of the five-year strategic plan?",
    "Did the company lose customers to the new APAC competitor?",
]

print("=" * 60)
print("Business Q&A Bot -- Full Pipeline")
print("=" * 60)

for q in questions:
    result = answer_question(q, all_chunks, client, MODEL_NAME)
    print(f"\nQuestion : {q}")
    print(f"\nAnswer   :")
    print(result["answer"])
    print(f"\nSources  : {', '.join(result['sources'])}")
    print("-" * 60)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 5 -- Evaluating Q&A quality
#
# Simple evaluation: check whether the answer contains the expected key fact.
# For production: extend with human review or LLM-as-judge scoring.
# ─────────────────────────────────────────────────────────────────────────────

eval_set = [
    {"question": "What was Q3 2024 revenue?",           "expected": "248",       "note": "Should mention $248M"},
    {"question": "What is the 2027 ARR target?",         "expected": "1.5",       "note": "Should mention $1.5B"},
    {"question": "What satisfaction score was achieved?","expected": "4.7",       "note": "Should mention 4.7/5.0"},
    {"question": "What was free cash flow in Q3?",       "expected": "42",        "note": "Should mention $42M FCF"},
]

print("=" * 60)
print("Q&A Quality Evaluation")
print("=" * 60)

passed = 0
for test in eval_set:
    result  = answer_question(test["question"], all_chunks, client, MODEL_NAME)
    found   = test["expected"].lower() in result["answer"].lower()
    status  = "PASS" if found else "FAIL"
    if found:
        passed += 1
    print(f"\n{status} | {test['question']}")
    print(f"  Expected : '{test['expected']}' -- {test['note']}")
    if not found:
        print(f"  Got      : {result['answer'][:120]}...")

print()
print(f"Score: {passed}/{len(eval_set)} ({100*passed//len(eval_set)}%)")
print()
print("Extend this with: human review, LLM-as-judge, tracking retrieval failures.")

### 📝 Exercise 7.6 — Capstone: Build Your Own Q&A Bot

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 7.6 -- Capstone: Bring your own corpus
#
# Your task:
#   1. Replace MY_CORPUS below with your own documents.
#      Options: a public annual report, industry white paper, news articles,
#               any plain text you have permission to use.
#
#   2. Write 5 questions you would genuinely want answered from those documents.
#
#   3. Run the pipeline and evaluate:
#        - Does the bot answer correctly?
#        - Does it cite the right source?
#        - Does it admit when it does not know?
#
#   4. Find at least ONE question that breaks the bot.
#      What would you change to fix it?
# ─────────────────────────────────────────────────────────────────────────────

MY_CORPUS = [
    {
        "source": "Document 1 -- replace with your source name",
        "text":   "Replace this with your actual document text. "
                  "Paste at least 200 words per document for useful results."
    },
    {
        "source": "Document 2 -- replace with your source name",
        "text":   "Add a second document here. More documents make the "
                  "retrieval challenge more interesting and realistic."
    },
]

MY_QUESTIONS = [
    "Your question 1 here?",
    "Your question 2 here?",
    "Your question 3 here?",
    "Your question 4 here?",
    "Your question 5 here?",
]

my_chunks = []
for doc in MY_CORPUS:
    my_chunks.extend(chunk_text(doc["text"], doc["source"]))

print(f"Corpus: {len(MY_CORPUS)} documents --> {len(my_chunks)} chunks")
print("=" * 60)

for q in MY_QUESTIONS:
    result = answer_question(q, my_chunks, client, MODEL_NAME)
    print(f"\nQuestion : {q}")
    print(f"Answer   : {result['answer']}")
    print(f"Sources  : {', '.join(result['sources']) if result['sources'] else 'None found'}")
    print("-" * 60)

---
## Versioning Your LLM Application

In Chapters 3–6, you saved everything needed to reproduce a trained model
into a `.pth` file using `ModelPipeline`.

In Chapter 7, there is no `.pth` file — but the same professional discipline applies.

| Component | What to save | Why |
|-----------|-------------|-----|
| **System prompt** | Exact text in a `.json` file | A one-word change can alter behaviour significantly |
| **Retrieval config** | `top_k`, chunk size, overlap | Changing these changes which context the model sees |
| **Model and version** | `"gpt-4o-mini"`, `"gemini-1.5-flash-002"` | Providers update models — pin the exact version |
| **Temperature** | The exact float | Affects output consistency |
| **Evaluation set** | Test questions and expected answers | Re-run checks after any change |

```python
# Save your configuration -- same discipline as saving a .pth file
import json

llm_config = {
    "version":       "1.0.0",
    "model":         "gpt-4o-mini",
    "temperature":   0.1,
    "top_k":         3,
    "chunk_size":    150,
    "chunk_overlap": 20,
    "system_prompt": "You are a financial analyst assistant...",
    "created_by":    "Your name",
    "created_date":  "2025-01-15",
}

with open("qa_bot_config_v1.json", "w") as f:
    json.dump(llm_config, f, indent=2)
print("LLM configuration saved.")
```

This habit separates an experiment from a reproducible professional tool.

---
## Chapter 7 Summary

| Concept | Key takeaway |
|---------|-------------|
| **What LLMs are** | Models pre-trained on vast text; you direct them, you do not train them |
| **From RNNs to Transformers** | Attention allows every token to relate to every other simultaneously |
| **Tokenization** | Text to token IDs; ~0.75 words per token; costs scale with token count |
| **Embeddings** | Tokens to dense vectors; similar meanings cluster together in space |
| **LLM APIs** | Send a prompt, receive a response; no GPU, no training, no model files |
| **Prompt engineering** | Specificity, roles, format examples, and temperature all shape quality |
| **Structured outputs** | Request JSON; parse programmatically; enables automated pipelines |
| **Conversation history** | LLMs have no memory -- you provide history in every API call |
| **Gradio** | One-function web app; makes LLM tools shareable instantly |
| **Hallucination** | LLMs state falsehoods confidently; always verify facts independently |
| **RAG** | Ground the model in your documents: retrieve context, then answer from context |
| **Versioning** | Save config, prompts, eval set -- the same reproducibility discipline as `.pth` |

---

## What You Have Built Across This Book

| Chapter | What you built | Key skill |
|---------|---------------|-----------|
| 1 | Perceptron · Training loop | Understanding how learning works |
| 2 | NumPy · pandas · PyTorch tensors | Working with data at every stage |
| 3 | Regression network · ModelPipeline | Training, saving, deploying |
| 4 | Churn classifier · Evaluation suite | Classification and business metrics |
| 5 | CNN defect detector | Image data and visual patterns |
| 6 | LSTM CO₂ forecaster | Sequences and time series |
| 7 | LLM Q&A bot | Directing pre-trained models |

The Bonus Chapter shows you how to take any of these models and serve them
as a live API endpoint — accessible by any application, any team, anywhere.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*
*Dr. M. Ramasubramaniam & Mr. Daniel Peter*

---
## Answer Key — Chapter 7 Exercises

---

### Exercise 7.1 — LLMs in Your Organisation (sample responses)

**Task 1 — Contract review:**
- Task: Legal team reads 30 supplier contracts per week flagging non-standard clauses
- LLM action: Flag unusual indemnification, liability cap, or exclusivity clauses with explanation
- Time saved: ~20 hours/week
- Risk: LLM misses a subtle clause; human review of flagged items is still required

**Task 2 — Customer email triage:**
- Task: Support manager reads 200 emails per day and assigns them to the right team
- LLM action: Classify and route emails by category, urgency, and sentiment
- Time saved: ~10 hours/week
- Risk: Misclassification sends an urgent complaint to the wrong team

**Task 3 — Competitor monitoring:**
- Task: Analyst reads 15 industry news sources daily and writes a weekly summary
- LLM action: Extract competitor mentions, generate structured weekly briefing
- Time saved: ~8 hours/week
- Risk: LLM may miss nuance or misinterpret technical announcements

---

### Exercise 7.2 — Tokenization

Typical observations:
- Technical and compound terms often split (e.g. "hyperparameter" splits into sub-words)
- Common abbreviations (ROI, KPI) tend to be single tokens in modern tokenizers
- Domain-specific rare words may increase token count by 20-40% over word count
- Implication: for domains with many rare terms, budget API costs with a 1.5x buffer

---

### Exercise 7.3 — Temperature comparison

Typical findings:
- Temperature 0.0 produces more concise, consistent, and structured answers
- Temperature 0.9 produces more varied phrasing and sometimes raises different points
- For business-critical analysis: prefer temperature 0.0-0.3
- For brainstorming or creative copy: temperature 0.7-1.0 is more useful
- Both runs should agree on key facts; significant factual disagreement signals a need to verify

---

### Exercise 7.5 — Hallucination

What you should have seen:
- The model produced plausible-sounding but invented citations, statistics, or regulations
- The tone was confident -- no hedging like "I'm not sure"
- This is not a bug -- the model generates the most probable next token, and "a plausible study"
  is statistically more probable than "I have no data on this"

Lesson: For any task where facts matter, use RAG or explicit verification steps.

---

### Exercise 7.6 — Capstone (self-assessment guide)

A well-functioning Q&A bot:
- Answers factual questions correctly when the fact appears in the corpus
- Cites the correct source document
- Admits uncertainty when the question cannot be answered from available text
- Fails gracefully rather than hallucinating

A bot that needs improvement typically:
- Retrieves wrong chunks (fix: improve keyword match or switch to embedding-based retrieval)
- Answers from memory rather than context (fix: strengthen the grounding instruction in system prompt)
- Gives verbose answers when short answers suffice (fix: add a length instruction to system prompt)